# 天龙人检测器 — 信号存在性验证

**目的**：用过去 1 年（2025-05-23 ~ 2026-05-22）的中概+科技股暴雷事件，验证 "期权异动 → 事件" 的统计命中率。

**Go/No-Go**：命中率 < 40% → 假设不成立，策略放弃。

**Only 3 数字**：样本总数 / 命中数 / 假阳性数。

In [ ]:
import sys, os, json, math, time
from datetime import date, datetime, timedelta
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from omega_quant.config import Settings, load_env_file
from omega_quant.adapters.massive_client import build_rest_client

settings = Settings()
load_env_file(settings.tiger_env_path)
client = build_rest_client(settings)
print('Massive client ready, API key present:', bool(settings.massive_api_key))

In [ ]:
# Trimmed universe for first-pass report: 5 stocks giving 7 events.
# Dropped COIN (8 events) because its 14596 contracts would take 3+ hours under server rate-limit.
# COIN can be added in a follow-up run overnight (cache resumes).
# Full original universe was 13; this run covers 7/18 events but produces a clean report tonight.
STOCKS = ['FUTU','NIO','PDD','BIDU','TAL']
DATE_START = '2025-05-23'
DATE_END   = '2026-05-22'
EVENT_THRESHOLD = 0.12
MARKET_TICKER = 'QQQ'
MARKET_THRESHOLD = 0.04
VOL_SPIKE_X = 3.0
PC_ZSCORE = 2.0
LOOKBACK_DAYS = [1, 2, 3]
CALM_NEXTDAY = 0.04
MAX_WORKERS = 20

REPORT_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)
print(f'Universe: {len(STOCKS)} tickers; window: {DATE_START} ~ {DATE_END}')

In [ ]:
def fetch_daily(ticker, start, end):
    aggs = list(client.list_aggs(ticker, 1, 'day', from_=start, to=end, adjusted=True, sort='asc', limit=400))
    rows = []
    for a in aggs:
        rows.append({
            'date': pd.Timestamp(a.timestamp, unit='ms', tz='UTC').strftime('%Y-%m-%d'),
            'open': a.open, 'high': a.high, 'low': a.low, 'close': a.close,
            'volume': int(a.volume or 0),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['date'])
    return df.set_index('date').sort_index()

stock_daily = {}
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = {ex.submit(fetch_daily, t, DATE_START, DATE_END): t for t in STOCKS + [MARKET_TICKER]}
    for f in as_completed(futs):
        t = futs[f]
        stock_daily[t] = f.result()
print({t: len(df) for t,df in stock_daily.items()})

In [ ]:
# Step 1: Identify event days
# IMPORTANT: use close-to-close return (captures overnight gaps), not intraday.
# Overnight gaps (earnings, news) are the majority of major events; intraday-only
# definition would miss FUTU 2026-05-22 (-34% gap), the canonical case.
qqq = stock_daily[MARKET_TICKER].copy()
qqq['qqq_c2c'] = qqq['close'].pct_change()

events = []
for t in STOCKS:
    df = stock_daily[t]
    if df.empty:
        continue
    c2c = df['close'].pct_change()                            # close-to-close (primary)
    intraday = (df['close'] - df['open']) / df['open']        # informational only
    gap = (df['open'] - df['close'].shift(1)) / df['close'].shift(1)
    for d, r in c2c.items():
        if pd.isna(r) or abs(r) <= EVENT_THRESHOLD:
            continue
        qm = qqq['qqq_c2c'].get(d, 0.0)
        if pd.notna(qm) and abs(qm) >= MARKET_THRESHOLD:
            continue   # systemic
        events.append({
            'ticker': t,
            'date': d,
            'event_return': float(r),
            'intraday_return': float(intraday.get(d, 0.0)),
            'gap_return': float(gap.get(d, 0.0)) if pd.notna(gap.get(d, np.nan)) else None,
            'qqq_return': float(qm) if pd.notna(qm) else None,
        })
events_df = pd.DataFrame(events).sort_values(['ticker','date']).reset_index(drop=True)
print(f'Events found: {len(events_df)}  (event_return = close-to-close)')
print(events_df.to_string())

In [ ]:
# Step 2: Fetch all options daily aggs per stock (concurrent, with retry + per-stock cache + strike filter)
EXPIRY_LTE = '2026-11-30'
RAW_CACHE_DIR = REPORTS_DIR / 'tianlongren_cache'
RAW_CACHE_DIR.mkdir(exist_ok=True)
MAX_WORKERS = 15

def strike_range(underlying):
    df = stock_daily[underlying]
    lo = float(df['low'].min()); hi = float(df['high'].max())
    return round(lo * 0.5, 2), round(hi * 2.0, 2)

def all_contracts_for(underlying, strike_lo, strike_hi):
    contracts = []
    for kw in [
        dict(underlying_ticker=underlying, expired=True,
             expiration_date_gte=DATE_START, expiration_date_lte=EXPIRY_LTE,
             strike_price_gte=strike_lo, strike_price_lte=strike_hi),
        dict(underlying_ticker=underlying, expired=False,
             expiration_date_lte=EXPIRY_LTE,
             strike_price_gte=strike_lo, strike_price_lte=strike_hi),
    ]:
        for attempt in range(4):
            try:
                contracts += list(client.list_options_contracts(**kw))
                break
            except Exception as e:
                if attempt == 3:
                    print(f'  list_options_contracts FAILED: {e}')
                    return contracts
                time.sleep(2 ** attempt)
    seen = set(); out = []
    for c in contracts:
        if c.ticker in seen: continue
        seen.add(c.ticker); out.append(c)
    return out

def fetch_contract_daily(c):
    for attempt in range(4):
        try:
            aggs = list(client.list_aggs(c.ticker, 1, 'day', from_=DATE_START, to=DATE_END,
                                         adjusted=True, sort='asc', limit=400))
            return [(pd.Timestamp(a.timestamp, unit='ms', tz='UTC').strftime('%Y-%m-%d'),
                     c.contract_type, int(a.volume or 0)) for a in aggs]
        except Exception:
            if attempt == 3:
                return []
            time.sleep(0.5 * (2 ** attempt))
    return []

def collect_stock(underlying):
    cache_file = RAW_CACHE_DIR / f'{underlying}.csv'
    if cache_file.exists():
        df = pd.read_csv(cache_file)
        print(f'{underlying}: cached {len(df)} rows', flush=True)
        return df.to_records(index=False).tolist()
    t0 = time.time()
    slo, shi = strike_range(underlying)
    contracts = all_contracts_for(underlying, slo, shi)
    n = len(contracts)
    print(f'{underlying}: {n} contracts (strike {slo}-{shi}), fetching aggs...', flush=True)
    rows = []
    done = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(fetch_contract_daily, c) for c in contracts]
        for f in as_completed(futs):
            for d, ct, v in f.result():
                rows.append((d, underlying, ct, v))
            done += 1
            if done % 500 == 0:
                print(f'  {underlying}: {done}/{n} contracts ({time.time()-t0:.0f}s elapsed)', flush=True)
    pd.DataFrame(rows, columns=['date','ticker','type','volume']).to_csv(cache_file, index=False)
    print(f'{underlying}: DONE  {n} contracts -> {len(rows)} daily rows  ({time.time()-t0:.1f}s)', flush=True)
    return rows

all_rows = []
for underlying in STOCKS:
    all_rows.extend(collect_stock(underlying))

opt_daily = pd.DataFrame(all_rows, columns=['date','ticker','type','volume'])
opt_daily['date'] = pd.to_datetime(opt_daily['date'])
print('Total rows:', len(opt_daily))

In [ ]:
# Step 3: Aggregate to daily ticker-level metrics
agg = opt_daily.groupby(['ticker','date','type'])['volume'].sum().unstack(fill_value=0)
agg.columns = [c if c in ('call','put') else c for c in agg.columns]
if 'call' not in agg.columns: agg['call'] = 0
if 'put'  not in agg.columns: agg['put']  = 0
agg['total'] = agg['call'] + agg['put']
agg['pc']    = agg['put'] / agg['call'].replace(0, np.nan)
agg = agg.reset_index().sort_values(['ticker','date'])

# 30-day rolling stats per ticker (use min_periods=10 for early dates)
agg['vol_ma30'] = agg.groupby('ticker')['total'].transform(
    lambda s: s.shift(1).rolling(30, min_periods=10).mean())
agg['pc_mean30'] = agg.groupby('ticker')['pc'].transform(
    lambda s: s.shift(1).rolling(30, min_periods=10).mean())
agg['pc_std30'] = agg.groupby('ticker')['pc'].transform(
    lambda s: s.shift(1).rolling(30, min_periods=10).std())

agg['vol_zscore'] = agg['total'] / agg['vol_ma30']
agg['pc_zscore']  = (agg['pc'] - agg['pc_mean30']) / agg['pc_std30']
print(agg.tail(20))

In [ ]:
# Step 4: Signal detection per event
agg_idx = agg.set_index(['ticker','date'])

def metric_at(ticker, d):
    try: return agg_idx.loc[(ticker, pd.Timestamp(d))]
    except KeyError: return None

def signal_triggered(row, direction):
    if row is None: return False
    vz = row.get('vol_zscore'); pz = row.get('pc_zscore')
    if pd.isna(vz) or pd.isna(pz): return False
    if vz <= VOL_SPIKE_X: return False
    if direction == 'down': return pz > PC_ZSCORE
    else: return pz < -PC_ZSCORE

results = []
for _, e in events_df.iterrows():
    direction = 'down' if e['event_return'] < 0 else 'up'
    df_t = agg[agg['ticker'] == e['ticker']].set_index('date')
    if df_t.empty:
        results.append({**e, 'direction': direction, 'hit': False, 'hit_offset': None}); continue
    event_ts = pd.Timestamp(e['date'])
    available = df_t.index[df_t.index < event_ts].sort_values()
    if len(available) < max(LOOKBACK_DAYS):
        results.append({**e, 'direction': direction, 'hit': False, 'hit_offset': None}); continue
    hit = False; hit_offset = None
    for off in LOOKBACK_DAYS:
        if off > len(available): break
        t_lookback = available[-off]
        row = df_t.loc[t_lookback]
        if signal_triggered(row, direction):
            hit = True; hit_offset = off; break
    results.append({**e, 'direction': direction, 'hit': hit, 'hit_offset': hit_offset})

hits_df = pd.DataFrame(results)
print(f'Events: {len(hits_df)}  Hits: {hits_df["hit"].sum()}  Hit rate: {hits_df["hit"].mean()*100:.1f}%')

In [ ]:
# Step 5: False positive rate on non-event days
event_set = set((r['ticker'], pd.Timestamp(r['date'])) for _, r in events_df.iterrows())

stock_returns = {}
for t in STOCKS:
    df = stock_daily[t]
    if df.empty: continue
    df = df.copy()
    df['next_ret'] = df['close'].pct_change().shift(-1)
    stock_returns[t] = df

triggers = []
for _, r in agg.iterrows():
    key = (r['ticker'], r['date'])
    if key in event_set: continue
    vz, pz = r['vol_zscore'], r['pc_zscore']
    if pd.isna(vz) or pd.isna(pz): continue
    if vz <= VOL_SPIKE_X or abs(pz) <= PC_ZSCORE: continue
    df_t = stock_returns.get(r['ticker'])
    if df_t is None: continue
    nxt = df_t['next_ret'].get(r['date'], None)
    if nxt is None or pd.isna(nxt): continue
    triggers.append({'ticker': r['ticker'], 'date': r['date'], 'vol_zscore': vz,
                     'pc_zscore': pz, 'next_ret': float(nxt),
                     'calm': abs(nxt) < CALM_NEXTDAY})

triggers_df = pd.DataFrame(triggers)
n_trig = len(triggers_df); n_calm = int(triggers_df['calm'].sum()) if n_trig else 0
fp_rate = (n_calm / n_trig * 100) if n_trig else 0.0
print(f'Non-event triggers: {n_trig}  False positives: {n_calm}  FP rate: {fp_rate:.1f}%')

In [ ]:
# Distribution plots: event-day T-1 metrics vs non-event-day metrics
tm1_records = []
for _, e in events_df.iterrows():
    df_t = agg[agg['ticker'] == e['ticker']].set_index('date')
    avail = df_t.index[df_t.index < pd.Timestamp(e['date'])].sort_values()
    if len(avail) >= 1:
        row = df_t.loc[avail[-1]]
        tm1_records.append({'vol_zscore': row['vol_zscore'], 'pc_zscore': row['pc_zscore']})
tm1_df = pd.DataFrame(tm1_records).dropna()

non_event = agg[~agg.apply(lambda r: (r['ticker'], r['date']) in event_set, axis=1)][['vol_zscore','pc_zscore']].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(non_event['vol_zscore'].clip(0, 10), bins=40, alpha=0.5, label='non-event', density=True)
axes[0].hist(tm1_df['vol_zscore'].clip(0, 10), bins=20, alpha=0.7, label='event T-1', density=True)
axes[0].axvline(VOL_SPIKE_X, color='red', ls='--', label=f'threshold={VOL_SPIKE_X}')
axes[0].set_xlabel('vol_zscore (vol/MA30)'); axes[0].set_ylabel('density'); axes[0].legend(); axes[0].set_title('Volume z-score')
axes[1].hist(non_event['pc_zscore'].clip(-5, 5), bins=40, alpha=0.5, label='non-event', density=True)
axes[1].hist(tm1_df['pc_zscore'].clip(-5, 5), bins=20, alpha=0.7, label='event T-1', density=True)
axes[1].axvline( PC_ZSCORE, color='red', ls='--'); axes[1].axvline(-PC_ZSCORE, color='red', ls='--')
axes[1].set_xlabel('pc_zscore'); axes[1].legend(); axes[1].set_title('P/C z-score')
plt.tight_layout()
plot_path = REPORTS_DIR / f'tianlongren_dist_{REPORT_TS}.png'
plt.savefig(plot_path, dpi=120)
print('Saved plot:', plot_path)

In [ ]:
# Step 6: Per-ticker breakdown + A/B group analysis + final markdown report
CHINA_ADR_GROUP = {'FUTU','PDD','NIO','BIDU','TAL','BABA','JD','XPEV','LI','GOTU'}
CONTROL_GROUP   = {'COIN','TSLA','NVDA'}

def group_of(ticker):
    if ticker in CHINA_ADR_GROUP: return 'China_ADR'
    if ticker in CONTROL_GROUP:   return 'Control_US'
    return 'Other'

hits_df['group'] = hits_df['ticker'].apply(group_of)
triggers_df['group'] = triggers_df['ticker'].apply(group_of) if len(triggers_df) else triggers_df

by_ticker = hits_df.groupby('ticker').agg(events=('hit','count'), hits=('hit','sum')).reset_index()
by_ticker['hit_rate'] = by_ticker['hits'] / by_ticker['events'].replace(0, np.nan)
by_ticker['group'] = by_ticker['ticker'].apply(group_of)

by_group = hits_df.groupby('group').agg(events=('hit','count'), hits=('hit','sum')).reset_index()
by_group['hit_rate'] = by_group['hits'] / by_group['events'].replace(0, np.nan)

# FP rate by group
if len(triggers_df):
    fp_by_group = triggers_df.groupby('group').agg(trig=('calm','count'), fp=('calm','sum')).reset_index()
    fp_by_group['fp_rate'] = fp_by_group['fp'] / fp_by_group['trig'].replace(0, np.nan)
else:
    fp_by_group = pd.DataFrame(columns=['group','trig','fp','fp_rate'])

N = len(hits_df); H = int(hits_df['hit'].sum())
hit_rate = (H/N*100) if N else 0.0

def grp_row(group_name):
    g = by_group[by_group['group']==group_name]
    if g.empty: return (0, 0, None)
    return (int(g['events'].iloc[0]), int(g['hits'].iloc[0]), float(g['hit_rate'].iloc[0]))

china_n, china_h, china_r = grp_row('China_ADR')
ctrl_n, ctrl_h, ctrl_r    = grp_row('Control_US')

def fmt_pct(x): return f'{x*100:.1f}%' if x is not None else 'pending'

lines = []
lines.append('# 天龙人检测器命中率研究报告（A/B 双组对照）')
lines.append(f'\n生成时间：{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
lines.append(f'回测区间：{DATE_START} ~ {DATE_END}')
lines.append(f'股票池：{", ".join(STOCKS)} ({len(STOCKS)} 支)')
lines.append('\n## 核心实验设计')
lines.append('- **维度 A — 中概股组**（FUTU/PDD/NIO/BIDU/TAL/...）：测试"中国跨境政策/监管内幕泄露"假设')
lines.append('- **维度 B — 美国高波控制组**（COIN/TSLA/NVDA）：测试加密/技术高 Beta 股是否有同样的提前定价行为')
lines.append('\n## 结论对照')
lines.append('| 组别 | 事件数 | 命中数 | 命中率 |')
lines.append('|------|------:|------:|------:|')
lines.append(f'| **维度 A 中概股**       | {china_n} | {china_h} | **{fmt_pct(china_r)}** |')
lines.append(f'| **维度 B 美国控制组**   | {ctrl_n}  | {ctrl_h}  | **{fmt_pct(ctrl_r)}** |')
if china_r is not None and ctrl_r is not None:
    diff = china_r - ctrl_r
    lines.append(f'| **差值 (A − B)**        | — | — | **{diff*100:+.1f}pp** |')
    lines.append('\n### 学术闭环判读')
    if china_r >= 0.5 and (ctrl_r is None or ctrl_r < 0.3):
        lines.append('✅ **强 A 弱 B**：中概股有明显的提前内幕脚印，美国本土高波股没有。"跨境信息不对称"假设成立。')
    elif china_r >= 0.5 and ctrl_r is not None and ctrl_r >= 0.5:
        lines.append('⚠️ **A=B 都强**：信号本质是"高 Beta 事件股的集体防御性定价"，不是中概股专属。仍是有效阿尔法。')
    elif china_r < 0.4:
        lines.append('❌ **中概股组命中率 <40%**：原假设不成立，策略放弃。')
    else:
        lines.append('🟡 **A 中等**：信号有效但不强，需要更多事件样本判断。')
else:
    lines.append('\n_⏳ 维度 B 数据待补充（COIN 期权数据未完成拉取）。维度 A 单独结论见下。_')

lines.append('\n## 维度 A 详情（中概股）')
lines.append(f'命中率 {fmt_pct(china_r)}（{china_h}/{china_n}）')
if china_r is not None:
    if china_r >= 0.5:
        lines.append('→ **>50% 阈值**：跨境信息不对称假设被强支持，进入下一阶段（添加 Sweep/Aggressiveness 滤波）。')
    elif china_r >= 0.4:
        lines.append('→ **40-50%**：临界，有效但不强；建议补 COIN 对照后再决策。')
    else:
        lines.append('→ **<40%**：假设证伪。')

lines.append(f'\n非事件日触发 {n_trig} 次，其中次日 |return| < {CALM_NEXTDAY*100:.0f}% 的"无功而返"占 **{fp_rate:.1f}%**')

lines.append('\n## 分股票拆解')
lines.append('| Ticker | Group | 事件数 | 命中数 | 命中率 |')
lines.append('|--------|-------|------:|------:|------:|')
for _, r in by_ticker.sort_values(['group','ticker']).iterrows():
    hr = f"{r['hit_rate']*100:.1f}%" if pd.notna(r['hit_rate']) else 'n/a'
    lines.append(f"| {r['ticker']} | {r['group']} | {int(r['events'])} | {int(r['hits'])} | {hr} |")

lines.append('\n## 命中事件明细')
hit_events = hits_df[hits_df['hit']].copy()
if len(hit_events):
    lines.append('| Ticker | Group | Event Date | Return | Dir | Hit Offset |')
    lines.append('|--------|-------|-----------|-------:|----:|----------:|')
    for _, r in hit_events.iterrows():
        lines.append(f"| {r['ticker']} | {r['group']} | {r['date'].strftime('%Y-%m-%d')} | {r['event_return']*100:+.1f}% | {r['direction']} | T-{int(r['hit_offset'])} |")
else:
    lines.append('（无命中）')

lines.append('\n## 漏检事件明细')
miss_events = hits_df[~hits_df['hit']].copy()
if len(miss_events):
    lines.append('| Ticker | Group | Event Date | Return | Dir |')
    lines.append('|--------|-------|-----------|-------:|----:|')
    for _, r in miss_events.iterrows():
        lines.append(f"| {r['ticker']} | {r['group']} | {r['date'].strftime('%Y-%m-%d')} | {r['event_return']*100:+.1f}% | {r['direction']} |")

lines.append(f'\n## 分布图\n\n![distribution]({plot_path.name})')
lines.append('\n## 阈值参数')
lines.append(f'- 事件阈值：|close-close| return > {EVENT_THRESHOLD*100:.0f}%')
lines.append(f'- 系统性过滤：QQQ |return| ≥ {MARKET_THRESHOLD*100:.0f}% 同日剔除')
lines.append(f'- 异动判定：vol/MA30 > {VOL_SPIKE_X}x AND |pc_zscore| > {PC_ZSCORE}（方向感知）')
lines.append(f'- 回看窗口：T-1, T-2, T-3 任一命中')
lines.append(f'- 假阳性定义：非事件日触发但次日 |return| < {CALM_NEXTDAY*100:.0f}%')
lines.append('\n## 数据限制')
lines.append('- Massive `list_aggs` 不返回 Open Interest 历史数据，本研究用 `Volume / 30日均值 > 3x` 作为新开仓代理')
lines.append('- 系统性日子用 QQQ |return| ≥ 4% 剔除，未做行业 beta 调整')
lines.append(f'- 原始数据：`tianlongren_raw_{REPORT_TS}.csv`，每股原始合约数据：`reports/tianlongren_cache/`')

report_md = '\n'.join(lines)
report_path = REPORTS_DIR / f'tianlongren_hit_rate_{REPORT_TS}.md'
report_path.write_text(report_md, encoding='utf-8')

raw_path = REPORTS_DIR / f'tianlongren_raw_{REPORT_TS}.csv'
agg.to_csv(raw_path, index=False)

print('Report saved:', report_path)
print('Raw CSV saved:', raw_path)
print('\n' + '='*60)
print(f'维度 A 中概股: {china_h}/{china_n} = {fmt_pct(china_r)}')
print(f'维度 B 美国: {ctrl_h}/{ctrl_n} = {fmt_pct(ctrl_r)}')
print(f'整体: {H}/{N} = {hit_rate:.1f}%  |  假阳性 {n_calm}/{n_trig} = {fp_rate:.1f}%')
print('='*60)